This notebook was tested under the new environment with Python 3.10 and following packages:
- drfp==0.3.7
- ipykernel==7.2.0
- numpy==2.2.6
- rdkit==2025.9.4
- scikit-learn==1.7.2
- pandas==2.3.3
- matplotlib==3.10.8

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error,r2_score
from sklearn.model_selection import train_test_split
from drfp import DrfpEncoder
random_seed = 42

## C–H arylation dataset

In [2]:
rxn_data = pd.read_csv("../dataset/rxn_data/aryl-scope-ligand.csv")
lig_smi_lst, rct1_smi_lst, rct2_smi_lst, pdt_smi_lst = rxn_data['ligand_smiles'].to_list(),rxn_data['electrophile_smiles'].to_list(),rxn_data['nucleophile_smiles'].to_list(),rxn_data['product_smiles'].to_list()
label = rxn_data['yield'].to_numpy()
len(set(lig_smi_lst)),len(rxn_data)

(24, 1536)

In [3]:
rxn_smi_with_lig_lst = [f"{rct1}.{rct2}.{lig}>>{pdt}" for rct1, rct2, lig, pdt in zip(rct1_smi_lst, rct2_smi_lst, lig_smi_lst, pdt_smi_lst)]
rxn_fp_arr_with_lig = np.array(DrfpEncoder.encode(rxn_smi_with_lig_lst))

rxn_smi_wo_lig_lst = [f"{rct1}.{rct2}>>{pdt}" for rct1, rct2, pdt in zip(rct1_smi_lst, rct2_smi_lst, pdt_smi_lst)]
rxn_fp_arr_wo_lig = np.array(DrfpEncoder.encode(rxn_smi_wo_lig_lst))

In [4]:
rxn_smi_with_lig_drfp_map = {smi:fp for smi, fp in zip(rxn_smi_with_lig_lst, rxn_fp_arr_with_lig)}
rxn_smi_wo_lig_drfp_map = {smi:fp for smi, fp in zip(rxn_smi_wo_lig_lst, rxn_fp_arr_wo_lig)}

In [5]:
np.save("./gen_desc/aryl_scope_rxn_smi_with_lig_drfp_map.npy", rxn_smi_with_lig_drfp_map)
np.save("./gen_desc/aryl_scope_rxn_smi_wo_lig_drfp_map.npy", rxn_smi_wo_lig_drfp_map)

## Asymmetric thiol addition dataset

In [6]:
rxn_data = pd.read_csv("../dataset/rxn_data/NS_acetal_dataset_with_pdt.csv")
rxn_data

,Unnamed: 0,Imine,Thiol,Catalyst,ΔΔG,Product
0,0,O=C(/N=C/c1ccccc1)c1ccccc1,Sc1ccccc1,O=P1(O)Oc2c(-c3ccccc3)cc3ccccc3c2-c2c(c(-c3ccc...,1.179891,O=C(NC(Sc1ccccc1)c1ccccc1)c1ccccc1
1,1,O=C(/N=C/c1ccccc1)c1ccccc1,CCS,O=P1(O)Oc2c(-c3ccccc3)cc3ccccc3c2-c2c(c(-c3ccc...,0.501759,CCSC(NC(=O)c1ccccc1)c1ccccc1
2,2,O=C(/N=C/c1ccccc1)c1ccccc1,SC1CCCCC1,O=P1(O)Oc2c(-c3ccccc3)cc3ccccc3c2-c2c(c(-c3ccc...,0.650584,O=C(NC(SC1CCCCC1)c1ccccc1)c1ccccc1
3,3,O=C(/N=C/c1ccccc1)c1ccccc1,COc1ccc(S)cc1,O=P1(O)Oc2c(-c3ccccc3)cc3ccccc3c2-c2c(c(-c3ccc...,1.238109,COc1ccc(SC(NC(=O)c2ccccc2)c2ccccc2)cc1
4,4,O=C(/N=C/c1ccc(C(F)(F)F)cc1)c1ccccc1,Sc1ccccc1,O=P1(O)Oc2c(-c3ccccc3)cc3ccccc3c2-c2c(c(-c3ccc...,1.179891,O=C(NC(Sc1ccccc1)c1ccc(C(F)(F)F)cc1)c1ccccc1
...,...,...,...,...,...,...
1070,1070,O=C(/N=C/c1ccccc1)c1ccccc1,Sc1ccccc1,O=P1(O)Oc2c(-c3cc(C(F)(F)F)cc(C(F)(F)F)c3)cc3c...,1.531803,O=C(NC(Sc1ccccc1)c1ccccc1)c1ccccc1
1071,1071,O=C(/N=C/c1ccccc1)c1ccccc1,Cc1ccccc1S,O=P1(O)Oc2c(-c3cc(C(F)(F)F)cc(C(F)(F)F)c3)cc3c...,1.531803,Cc1ccccc1SC(NC(=O)c1ccccc1)c1ccccc1
1072,1072,O=C(/N=C/c1ccc(C(F)(F)F)cc1)c1ccccc1,Cc1ccccc1S,O=P1(O)Oc2c(-c3cc(C(F)(F)F)cc(C(F)(F)F)c3)cc3c...,1.370104,Cc1ccccc1SC(NC(=O)c1ccccc1)c1ccc(C(F)(F)F)cc1
1073,1073,O=C(/N=C/c1cccc2ccccc12)c1ccccc1,Sc1ccccc1,O=P1(O)Oc2c(-c3cc(C(F)(F)F)cc(C(F)(F)F)c3)cc3c...,1.301167,O=C(NC(Sc1ccccc1)c1cccc2ccccc12)c1ccccc1


In [7]:
imine_lst = rxn_data['Imine'].to_list()
thiol_lst = rxn_data['Thiol'].to_list()
cat_lst = rxn_data['Catalyst'].to_list()
pdt_lst = rxn_data['Product'].to_list()
label = rxn_data['ΔΔG'].to_numpy()

In [8]:
rxn_smi_with_cat_lst = [f"{imine}.{thiol}.{cat}>>{pdt}" for imine,thiol,cat,pdt in zip(imine_lst,thiol_lst,cat_lst,pdt_lst)]
rxn_fp_arr_with_cat = np.array(DrfpEncoder.encode(rxn_smi_with_cat_lst))

rxn_smi_wo_cat_lst = [f"{imine}.{thiol}>>{pdt}" for imine,thiol,pdt in zip(imine_lst,thiol_lst,pdt_lst)]
rxn_fp_arr_wo_cat = np.array(DrfpEncoder.encode(rxn_smi_wo_cat_lst))

In [9]:
rxn_smi_with_cat_drfp_map = {smi:fp for smi, fp in zip(rxn_smi_with_cat_lst, rxn_fp_arr_with_cat)}
rxn_smi_wo_cat_drfp_map = {smi:fp for smi, fp in zip(rxn_smi_wo_cat_lst, rxn_fp_arr_wo_cat)}

In [ ]:
np.save("./gen_desc/thiol_add_rxn_smi_with_cat_drfp_map.npy", rxn_smi_with_cat_drfp_map)
np.save("./gen_desc/thiol_add_rxn_smi_wo_cat_drfp_map.npy", rxn_smi_wo_cat_drfp_map)